## 16. تشخیص Duplicate

آگهی‌های تکراری می‌توانند حجم عرضه را بیش از واقع نشان دهند.

تیم باید حداقل دو سطح Duplicate را بررسی کند:

1. **Exact Duplicate:** رکوردهای کاملاً یکسان
2. **Probable Duplicate:** آگهی‌های احتمالاً مربوط به یک ملک

ویژگی‌های احتمالی برای Duplicate تقریبی:

- شهر و محله
- مختصات نزدیک
- مساحت
- تعداد اتاق
- طبقه
- قیمت مشابه
- متن مشابه
- ماه ثبت
- نوع کاربر

حذف Duplicate احتمالی باید محافظه‌کارانه و قابل Audit باشد. در صورت عدم حذف، اثر آن بر
شاخص عرضه باید در تحلیل حساسیت بررسی شود.

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from itertools import combinations
import hashlib

from datasets import load_dataset

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [4]:
df = pd.read_feather("../Outputs/02_df.feather")

In [5]:
has_location = (
    df["location_latitude"].notna() &
    df["location_longitude"].notna()
)

In [6]:
df["is_exact_duplicate"] = df.duplicated(keep="first")

print("Exact duplicates:", df["is_exact_duplicate"].sum())

Exact duplicates: 11


In [7]:
df["lat_key"] = np.nan
df["lon_key"] = np.nan

df["lat_key"] = df["location_latitude"].round(4)
df["lon_key"] = df["location_longitude"].round(4)

In [ ]:
duplicate_cols = [
    "city_slug",
    "neighborhood_slug",
    "cat3_slug",
    "property_type",
    "price_regime",
    "lat_key",
    "lon_key",
    "building_size",
    "land_size",
    "rooms_count",
    "floor",
    "unit_per_floor"
]



In [16]:
df["duplicate_group"] = np.nan

df.loc[has_location, "duplicate_group"] = (
    df.loc[has_location]
      .groupby(duplicate_cols, dropna=False)
      .ngroup()
)

group_size = (
    df.groupby("duplicate_group")["duplicate_group"]
      .transform("size")
)

df["is_probable_duplicate"] = (
    has_location &
    (group_size > 1)&
    (
        df["unit_per_floor"].isna() |
        (group_size > df["unit_per_floor"])
    )
)

print(
    "Probable duplicate rows:",
    df["is_probable_duplicate"].sum()
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_2380\2649463317.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(duplicate_cols, dropna=False)


Probable duplicate rows: 8598


In [17]:
# ============================================
# 11. Duplicate statistics
# ============================================

total_rows = len(df)

duplicate_rows = df["is_probable_duplicate"].sum()

non_duplicate_rows = total_rows - duplicate_rows

print("=" * 50)
print("Duplicate Detection Results")
print("=" * 50)

print("Total rows:", total_rows)
print("Rows with valid location:", has_location.sum())
print("Rows without location:", (~has_location).sum())
print("Probable duplicate rows:", duplicate_rows)
print("Non-duplicate rows:", non_duplicate_rows)

Duplicate Detection Results
Total rows: 999953
Rows with valid location: 655586
Rows without location: 344367
Probable duplicate rows: 8598
Non-duplicate rows: 991355


In [21]:
# ============================================
# 13. View duplicate records
# ============================================

duplicates = (
    df[df["is_probable_duplicate"]]
    .sort_values("duplicate_group")
)

duplicate_view = duplicates[
    [
        "duplicate_group",
        "city_slug",
        "neighborhood_slug",
        "cat3_slug",
        "property_type",
        "price_regime",
        "building_size",
        "land_size",
        "rooms_count",
        "floor",
        "unit_per_floor",
        "location_latitude",
        "location_longitude",
        "title",
        "description"
    ]
]

duplicate_view.to_csv("../Outputs/probable_duplicates.csv")

In [ ]:
df["is_probable_duplicate_to_remove"] = False

duplicate_mask = df["is_probable_duplicate"]

df.loc[duplicate_mask, "is_probable_duplicate_to_remove"] = (
    df.loc[duplicate_mask]
      .duplicated(
          subset=duplicate_cols,
          keep="first"
    )
)

print(
    "Rows to remove:",
    df["is_probable_duplicate_to_remove"].sum()
)

Rows to remove: 4455


In [ ]:
# ============================================
# 17. Remove temporary columns
# ============================================

temporary_columns = [
    "lat_key",
    "lon_key",
]

df = df.drop(
    columns=temporary_columns,
    errors="ignore"
)

print("Final shape:", df.shape)

Final shape: (993927, 65)


In [ ]:
df.to_feather(
    "../Outputs/03_df.feather"
)